# Omnigrok modular addition: p=211 with fixed weight decay

The `weight_decay=0.30` run memorized quickly, but validation accuracy was still only about 2% after 50k steps. This version keeps the same data split, architecture, optimizer, learning rate, and batch size, and changes only the constant AdamW decay to `1.0`.


## Protocol

For p=211 the training set contains 34 examples per residue/class. AdamW uses fixed `weight_decay=1.0` from the first update. There is no delayed activation and no phase-dependent optimizer change. The new protocol name prevents old TensorBoard data or checkpoints from being mixed with this run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/grokking_prediction_original/2026-Project-202/code/Grokking/modular_addition_grokking_colab

In [ ]:
!pip install -q tensorboard

In [ ]:
from prime_sweep_omnigrok import Config, run_sweep

cfg = Config(
    primes=(211,),
    output_root='/content/drive/MyDrive/grokking_prime_sweep',
    protocol_name='omnigrok_p211_v9_fixed_wd100',

    model_dtype='float32',
    fused_adamw=True,
    batch_size=512,
    batch_size_by_p={211: 512},
    learning_rate=1e-3,

    # Keep the split that produced a clear memorization phase.
    normalize_train_examples_per_class=True,
    train_examples_per_class=34.0,

    # One constant regularization strength for the entire run.
    delayed_weight_decay=False,
    weight_decay=1.0,
    weight_decay_by_p={211: 1.0},

    max_steps=100_000,
    log_every=50,
    text_log_every=1_000,
    required_gap_steps=5_000,
    post_grok_steps=5_000,
    text_log_enabled=True,
    force_restart=True,
)
CONFIG = cfg
cfg


## Live TensorBoard

Start the next cell before training. Scalars include loss/accuracy, phase, fixed random projections, and Optimization/weight_decay. The text log prints a phase summary every 1,000 steps.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/grokking_prime_sweep --reload_interval 5

## Expected behavior and stopping rule

Train accuracy should again reach approximately 1 first. The stronger constant decay should make validation accuracy turn upward much earlier than in the WD=0.30 run. After stable validation accuracy reaches 0.95 and the measured gap is at least 5,000 steps, the code records another 5,000 post-grok steps and stops automatically.


In [ ]:
summaries = run_sweep(cfg)